# Level 1 · Step 6 — Build one business memory per business (Colab Pro L4)

**Goal:** turn each business's own messages ("evidence", built locally by `src/data/06_collect_evidence.py`)
into one **business memory** in our format (`config/memory_schema.json`): name, category, about, services +
prices, timings, locations, contact, FAQs, policies, offers, tone, hand-over message. These memories are what
the fine-tuned model learns to answer from (step 7), exactly like a Chatrace customer's Memory form later.

## What this notebook is — and is not

- **IS:** read-only extraction with an open teacher model (Qwen2.5 Instruct, Apache-2.0). Nothing is trained here.
- **IS:** resumable. Every shard (50 businesses) is pushed to the private dataset repo
  `ankit-iiitdmj/mbg-memories-<export>` before the next one starts.
- **IS NOT:** fine-tuning. That is Level 3, a separate notebook.

## GPU

Pick **L4** (Runtime → Change runtime type). The teacher is chosen from the GPU automatically:
T4 → 7B · **L4 → 14B** · A100 → 32B. Expected time on the L4: **about 1 hour** for ~1,000 businesses.
Disconnect the runtime when Cell 11 has finished — an idle GPU still uses compute units.

## First-shard health check

After the FIRST shard of each session the run aborts if more than 25% of teacher answers are unusable, or the
memories are almost empty — so a broken prompt never burns compute units.

In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL  = "https://github.com/ankit8453/chat-AI-model.git"   # private repo -> asks for a GitHub token once
REPO_PATH = "/content/chat-AI-model"

import os, subprocess, sys, getpass

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", url, REPO_PATH], capture_output=True, text=True)

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "README.md")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    r = _clone(REPO_URL)
    if r.returncode != 0:
        print("Public clone failed (private repo?). Paste a GitHub token with read access:")
        tok = getpass.getpass("GitHub token: ")
        r = _clone(REPO_URL.replace("https://", f"https://{tok}@"))
        if r.returncode != 0:
            raise SystemExit("git clone failed:\n" + r.stderr[-1500:])
        subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", REPO_URL], check=False)
else:
    # Already cloned in this runtime -> pull the latest code (fixes pushed since the last run).
    r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], capture_output=True, text=True)
    if r.returncode != 0:
        tok = getpass.getpass("GitHub token (to pull updates): ")
        r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only",
                            REPO_URL.replace("https://", f"https://{tok}@"), "main"], capture_output=True, text=True)
    print("git pull:", (r.stdout or r.stderr).strip()[-300:])

print("code version:", subprocess.run(["git", "-C", REPO_PATH, "log", "-1", "--oneline"],
                                      capture_output=True, text=True).stdout.strip())
os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "src"))
print("Repo root contents:", sorted(os.listdir(".")))

_pip_packages = ["huggingface_hub>=0.24", "transformers>=4.45", "accelerate>=0.33", "bitsandbytes>=0.43"]
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pip_packages], capture_output=True, text=True)
if proc.returncode != 0:
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tail above.")

# vLLM is optional: much faster, but newer releases may not support the T4. If it fails we fall back.
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
print("vllm install:", "ok" if proc.returncode == 0 else "FAILED (will use transformers 4-bit fallback)")
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
HF_NS = _whoami["name"]
print(f"HF Hub ok: user={HF_NS}")

In [ ]:
# Cell 4 — GPU check + teacher config (model, context length, shard size)
import subprocess
subprocess.run(["nvidia-smi"], check=False)
print()

from judge.engine import memory_config
cfg = memory_config()
for k, v in cfg.items():
    print(f"  {k}: {v}")

## Memory configuration

- Export: **`messages_2026-10-31`** (change `EXPORT` in Cell 6 for later exports)
- Input: `ankit-iiitdmj/mbg-chat-candidates` → `<EXPORT>/business_evidence.jsonl` (PII-redacted, one row per business)
- Teacher: **Qwen2.5-14B-Instruct-AWQ** on the L4 via vLLM (7B on a T4), greedy decoding, max 1,500 new tokens
- Prompt + parser: `src/memory/prompt.py` — facts copied only from the business's own messages, newest wins
- Output: `ankit-iiitdmj/mbg-memories-<EXPORT>` → `shards/shard_XXXXXX.jsonl`, then `memories_all.jsonl` + `summary.json`

In [ ]:
# Cell 6 — load business evidence from the private HF dataset
import json
from huggingface_hub import hf_hub_download

EXPORT = "messages_2026-10-31"
_p = hf_hub_download(f"{HF_NS}/mbg-chat-candidates", f"{EXPORT}/business_evidence.jsonl", repo_type="dataset")
with open(_p, encoding="utf-8") as fh:
    evidence = [json.loads(l) for l in fh if l.strip()]
evidence.sort(key=lambda e: -e["kept_questions"])      # businesses with the most real questions first

print(f"businesses: {len(evidence):,}")
print(f"snippets:   {sum(len(e['snippets']) for e in evidence):,}")
print(f"chars p50:  {sorted(e['chars'] for e in evidence)[len(evidence)//2]:,}")

In [ ]:
# Cell 7 — resume state from HF Hub (shards already built in earlier sessions)
from judge.hub import JudgeShardManager

ckpt_mgr = JudgeShardManager(HF_NS, EXPORT, prefix="mbg-memories")
done_rows, next_shard = ckpt_mgr.try_load_done()
done_ids = {r["business_id"] for r in done_rows}
remaining = [e for e in evidence if e["business_id"] not in done_ids]

print(f"repo: {ckpt_mgr.repo_id} (private)")
print(f"already built: {len(done_rows):,} in {next_shard} shards")
print(f"remaining:     {len(remaining):,}")
if not remaining:
    print("Nothing left to build — go to Cell 10 (retry) / Cell 11 (summary).")

In [ ]:
# Cell 8 — build the teacher engine (vLLM, else transformers 4-bit)
from judge.engine import build_engine
engine = build_engine(cfg, backend="auto")

In [ ]:
# Cell 9 — memory loop: one shard at a time, pushed to HF before the next; first-shard health check
import time, datetime
from memory.prompt import build_messages_fitted, parse_memory, memory_fill, health_check

SHARD = cfg["shard_size"]
MAX_SHARDS_THIS_SESSION = None   # e.g. 2 to stop early on purpose; None = run until done / disconnected

def build_rows(batch, shrink=1.0):
    fitted = []
    for e in batch:
        ev = e if shrink == 1.0 else {**e, "snippets": e["snippets"][:max(1, int(len(e["snippets"]) * shrink))]}
        fitted.append(build_messages_fitted(ev, engine.count_tokens, engine.max_input_tokens))
    outputs = engine.generate_safe([m for m, _ in fitted])   # never crashes the shard
    rows = []
    for e, (_, used), text in zip(batch, fitted, outputs):
        m = parse_memory(text) if text is not None else None
        rows.append({
            "business_id": e["business_id"], "kept_questions": e["kept_questions"], "memory": m,
            "fill": memory_fill(m), "snippets_used": used, "snippets_total": len(e["snippets"]),
            "raw": None if m else ("<generation failed>" if text is None else text[:500]),
            "engine": engine.name, "model": engine.model_id,
            "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        })
    return rows

t_start = time.time(); built_now = 0; shard_idx = next_shard
for s, i in enumerate(range(0, len(remaining), SHARD)):
    if MAX_SHARDS_THIS_SESSION is not None and s >= MAX_SHARDS_THIS_SESSION:
        print("Reached MAX_SHARDS_THIS_SESSION — stopping cleanly."); break
    batch = remaining[i:i + SHARD]
    t0 = time.time()
    rows = build_rows(batch)
    dest = ckpt_mgr.save_shard(shard_idx, rows)
    if s == 0:
        health_check(rows)
    built_now += len(rows); shard_idx += 1
    rate = built_now / max(time.time() - t_start, 1e-6)
    left = len(remaining) - (i + len(batch))
    ok = sum(r["memory"] is not None for r in rows); fill = sum(r["fill"] for r in rows) / len(rows)
    print(f"[shard {shard_idx-1}] {len(rows)} businesses in {time.time()-t0:.0f}s | ok {ok}/{len(rows)} | "
          f"avg sections {fill:.1f} | left {left:,} (~{left/max(rate,1e-6)/60:.0f} min) | pushed {dest}")

print(f"\nThis session built {built_now:,} memories.")

## Retry the failures once

Businesses whose memory could not be parsed are tried once more with **half** the evidence (a shorter prompt
often fixes a broken or cut-off JSON). Results go into new shards; the newest row per business wins in Cell 11.

In [ ]:
# Cell 10 — retry unusable memories once with half the evidence
all_rows, shard_idx = ckpt_mgr.try_load_done()
latest = {}
for r in all_rows:
    latest[r["business_id"]] = r
failed_ids = {b for b, r in latest.items() if r["memory"] is None and not r.get("retried")}
retry = [e for e in evidence if e["business_id"] in failed_ids]
print(f"retrying {len(retry)} businesses")
for i in range(0, len(retry), SHARD):
    rows = build_rows(retry[i:i + SHARD], shrink=0.5)
    for r in rows:
        r["retried"] = True
    dest = ckpt_mgr.save_shard(shard_idx, rows); shard_idx += 1
    print(f"retry shard: ok {sum(r['memory'] is not None for r in rows)}/{len(rows)} | pushed {dest}")

## Summary + verdict

Re-loads **all** shards from the Hub (newest row per business wins), writes `memories_all.jsonl` and
`summary.json` to `ankit-iiitdmj/mbg-memories-<EXPORT>`, prints 3 example memories, and gives the verdict.

In [ ]:
# Cell 11 — aggregate all shards, push memories_all.jsonl + summary.json, show examples, verdict
import json, random, tempfile, os
from collections import Counter

all_rows, _ = ckpt_mgr.try_load_done()
latest = {}
for r in all_rows:
    latest[r["business_id"]] = r
rows = list(latest.values())
good = [r for r in rows if r["memory"] is not None and r["fill"] >= 3]

summary = {
    "export": EXPORT, "businesses": len(evidence), "built": len(rows),
    "unusable": sum(1 for r in rows if r["memory"] is None),
    "thin (1-2 sections)": sum(1 for r in rows if r["memory"] is not None and r["fill"] < 3),
    "good (3+ sections)": len(good),
    "kept_questions_covered_by_good": sum(r["kept_questions"] for r in good),
    "sections_filled": dict(Counter(k for r in rows if r["memory"] for k in r["memory"]).most_common()),
    "categories_top20": dict(Counter(r["memory"].get("category", "?") for r in good).most_common(20)),
    "with_handover_message": sum(1 for r in good if r["memory"].get("handover_message")),
    "teacher": rows[0]["model"] if rows else None,
}
tmp = tempfile.mkdtemp()
with open(os.path.join(tmp, "memories_all.jsonl"), "w", encoding="utf-8") as fh:
    for r in rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
with open(os.path.join(tmp, "summary.json"), "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)
ckpt_mgr.save_file(os.path.join(tmp, "memories_all.jsonl"), "memories_all.jsonl", "all business memories")
ckpt_mgr.save_file(os.path.join(tmp, "summary.json"), "summary.json", "memory summary")
print(json.dumps(summary, indent=2, ensure_ascii=False))

random.seed(0)
for r in random.sample(good, min(3, len(good))):
    print(f"\n=== {r['business_id']} · {r['kept_questions']} real questions · {r['fill']} sections ===")
    print(json.dumps(r["memory"], indent=1, ensure_ascii=False)[:2500])

done = len(rows) >= len(evidence)
print("\nProgress:", f"{len(rows):,}/{len(evidence):,} built", "(complete)" if done else "(run Cell 9 again)")
print("VERDICT:", "READY for training examples (step 7)" if len(good) >= 600 else
      ("NOT DONE YET — run Cell 9 again" if not done else "TOO FEW GOOD MEMORIES — send the summary to Claude"))

## Decision tree

- **READY for training examples** — at least 600 businesses have a good memory (3+ sections). Send the Cell 11
  output to Claude; step 7 builds the training examples (memory + real question → answer, plus hand-over and
  language-match examples).
- **NOT DONE YET** — the session ended early. Run all again: Cell 7 resumes from the Hub.
- **TOO FEW GOOD MEMORIES / ABORT in health check** — send the output to Claude; the prompt in
  `src/memory/prompt.py` gets fixed, pushed to GitHub, and you Run all again.